# Abertura P × Q — Enel Distribuição Rio (RJ) e Enel Distribuição Ceará (CE)

Este notebook abre a receita de fornecimento por item tarifário e por família, a partir da tarifa
homologada pela ANEEL. É o mesmo método de SP; muda a fonte dos dados. O porquê de cada transformação
está em [`procedimentos/PROCEDIMENTO_PXQ.md`](procedimentos/PROCEDIMENTO_PXQ.md); o que muda em relação a
SP está na seção 0 do [README](../README.md).

A saída traz dois blocos de valores lado a lado, como em SP:

| Bloco | O que é | Para que serve |
|---|---|---|
| **1 — Calculado** | quantidade × tarifa homologada cheia, nos 37 itens e nas 4 camadas | É a régua: mostra o que a tarifa explica sozinha |
| **2 — Ajustado** | o bloco 1 corrigido até fechar na receita faturada | É o produto final: a receita aberta por item |

**O caminho:**

- **Parte I — Clientes livres e geradoras (base SAMP LV).** Ler a base e conferir → separar o que é
  tarifa → montar a chave → buscar a tarifa → definir a quantidade → decidir quais parcelas incidem →
  multiplicar (bloco 1) → agrupar → ajustar (bloco 2) → conferir.
- **Parte II — Clientes cativos (base SAMP de documentos).** A construir, quando a base completa chegar.
- **Saídas.** `pxq_<UF>_<ref>.xlsx` e `PxQ_regulacao_<UF>_<ref>.xlsx`, com as mesmas abas e colunas de SP.

---
## Passo 0 — Parâmetros

Tudo o que muda de um mês para outro está nesta célula. Todo mês mudam só `DISTRIBUIDORA` e
`REFERENCIA`; os arquivos de origem seguem o nome padrão de cada entrega.

In [ ]:
import os
import re
import datetime as dt
from calendar import monthrange
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------- folders
ROOT = Path(os.environ.get("PXQ_ROOT", r"C:\python_projects\claude\p_x_q"))
BASE = ROOT / "CE_RJ" / "FAT"

# ---------------------------------------------------------------- period and distributor
DISTRIBUIDORA = "RJ"            # RJ or CE
REFERENCIA    = "2026-10"       # accounting month (AAAA-MM)

# Common tariff table (all distributors and vintages), built by common/3-Py/extrai_pcat.py
ARQ_TARIFA_LONG = ROOT / "common" / "tbl" / "tarifas" / "tarifa_long.csv"
# Subclass -> consumption class (same de-para used by the RNF of CE and RJ)
ARQ_CLASSE = ROOT / "CE_RJ" / "RNF" / "docs" / "CL_SUB_TAR.xlsx"
# Free customers and generators: SAMP LV extraction of the month (one row per document)
ARQ_LV = BASE / "src" / "LV" / f"{REFERENCIA[5:7]}.{REFERENCIA[:4]}_SAMP_{DISTRIBUIDORA} LV.XLSX"
# Management report of the month (check of the free-market energy)
ARQ_INFORME = BASE / "2-Checks" / "Informe_Gestao" / (
    f"Informe_de_Gestão_{REFERENCIA[:4]}{REFERENCIA[5:7]}_(Enel Distribuição "
    f"{'RJ' if DISTRIBUIDORA == 'RJ' else 'CE'}).xlsb")

# 2nd pass (Passo 9). True = block 2 closes exactly on the billed revenue (standard delivery).
APLICAR_2A_PASSADA = True

SAIDA = BASE / "1-Entrega" / "pxq" / (f"pxq_{DISTRIBUIDORA}_{REFERENCIA}.xlsx" if APLICAR_2A_PASSADA
                                      else f"pxq_{DISTRIBUIDORA}_{REFERENCIA}_sem_2a_passada.xlsx")
SAIDA_REGULACAO = BASE / "1-Entrega" / "pxq_regulacao" / (
    f"PxQ_regulacao_{DISTRIBUIDORA}_{REFERENCIA}.xlsx" if APLICAR_2A_PASSADA
    else f"PxQ_regulacao_{DISTRIBUIDORA}_{REFERENCIA}_sem_2a_passada.xlsx")

# Exception list (Passo 7b): lines without direct tariff association and discount divergences, every month,
# for the billing team
SAIDA_EXCECOES = BASE / "2-Checks" / "lista_excecoes" / f"lista_excecoes_{DISTRIBUIDORA}_{REFERENCIA}.xlsx"
# tolerance of the discount check: a document is listed when the difference between the discount by tariff
# and the informed discount exceeds BOTH limits: a share of the calculated discount and a minimum in R$.
# AT = group A documents (free customers and AT captive); BT = low-voltage captive documents (Part II)
TOL_DESCONTO_PCT = 0.005          # 0,5% of the calculated discount, per document
TOL_DESCONTO_MIN_AT_BRL = 10.0    # minimum absolute difference, group A
TOL_DESCONTO_MIN_BT_BRL = None    # to be defined with the captive analysis (Part II)

print("distribuidora:", DISTRIBUIDORA, "| referencia:", REFERENCIA)
print("base livres  :", ARQ_LV.name)
print("saida        :", SAIDA.name)
print("2a passada   :", "aplicada" if APLICAR_2A_PASSADA else "DESLIGADA")

---
# Parte I — Clientes livres e geradoras (base SAMP LV)

## Passo 1 — Ler a base e conferir o recebimento

A base dos livres é a extração SAMP LV do mês: **uma linha por documento emitido** (fatura), com a
conta contrato, a categoria tarifária (`Ctg.tar.`, por exemplo `A4_LVVD` = A4 livre Verde), a subclasse
(`ZnTemp.`) e, em colunas, a energia e a demanda por posto com a receita de cada uma, sem tributos.

O que a leitura faz:

- lê só as linhas que têm número de documento (as linhas de conferência que alguém tenha escrito no fim
  da planilha ficam de fora);
- guarda o número da linha no Excel (`LINHA_EXCEL`), para rastrear qualquer valor até a origem;
- identifica o tipo de faturamento pelo `Código Arquivo`: 05 = faturamento (FAT), 06 = refaturamento
  (REFAT). No Passo 2, o refaturamento é separado em `REFAT estorno` (quantidade negativa) e `REFAT conta
  nova` (positiva): juntos, as quantidades se anulam e a receita não (por exemplo, quando o refaturamento
  retira um desconto), e o grupo ficaria com receita sem quantidade para valorizar.

**Conferência.** A energia faturada (ponta + fora ponta) é comparada com a linha "Consumo Faturado
(Livres)" do Informe de Gestão do mês. A diferença esperada é a energia faturada fora da base SAMP LV
(suprimento à Elektro, no RJ).

In [ ]:
# Free-market SAMP columns come prefixed with the SAP table name; the prefix is dropped
lv = pd.read_excel(ARQ_LV, dtype={"Nº doc.": str, "Código Arquivo": str, "Cta.contrato": str})
lv.columns = [c.replace("ZISU_T_SAMP_M_LV-", "") for c in lv.columns]
lv["LINHA_EXCEL"] = lv.index + 2                         # header is Excel row 1
lv = lv[lv["Nº doc."].notna()].copy()                    # drops manual check rows at the bottom

COLUNAS_NUM = ["CONS_PONTA_TUSD", "CONS_FPONTA_TUSD", "CONS_RESERVADO_TUSD", "CONS_INTERMED_TUSD",
               "CONS_NSA_TUSD", "RECEITA_PONTA", "RECEITA_FPONTA", "RECEITA_RESERVADO",
               "RECEITA_INTERMED", "RECEITA_NSA", "TOTAL_FAT", "ICMS", "PIS", "COFINS",
               "RECEITA_BANDEIRA", "DEM_PONTA", "DEM_FPONTA", "DEM_NSA", "ULT_DEM_PONTA",
               "ULT_DEM_FPONTA", "ULT_DEM_NSA", "RECEITA_DEM_PONTA", "RECEITA_DEM_FPONTA",
               "RECEITA_DEM_NSA", "RECEITA_ULT_DEM_PONTA", "RECEITA_ULT_DEM_FPONTA",
               "RECEITA_ULT_DEM_NSA", "SUBSCARFI", "SUBSGERFI", "SUBSDIST"]
for c in COLUNAS_NUM:
    lv[c] = pd.to_numeric(lv[c], errors="raise").fillna(0.0)

TIPO_ARQUIVO = {"05": "FAT", "06": "REFAT"}
lv["TIPO_FATURAMENTO"] = lv["Código Arquivo"].map(TIPO_ARQUIVO)
sem_tipo = lv["TIPO_FATURAMENTO"].isna()
if sem_tipo.any():
    raise ValueError(f"Codigo Arquivo sem de-para: {sorted(lv.loc[sem_tipo, 'Código Arquivo'].unique())}")

energia_kwh = lv[["CONS_PONTA_TUSD", "CONS_FPONTA_TUSD", "CONS_INTERMED_TUSD",
                  "CONS_RESERVADO_TUSD", "CONS_NSA_TUSD"]].sum().sum()
print(f"{len(lv)} documentos | {lv['Cta.contrato'].nunique()} contas contrato | "
      f"{lv['Data'].min():%d/%m/%Y} a {lv['Data'].max():%d/%m/%Y}")
print(lv.groupby("TIPO_FATURAMENTO").size().rename("documentos").to_string())
print(f"energia faturada: {energia_kwh / 1e3:,.0f} MWh | total faturado: "
      f"{lv['TOTAL_FAT'].sum() / 1e6:,.2f} MR$")

In [ ]:
# Check against the management report: line "Consumo Faturado (Livres) - KWh", month of REFERENCIA
from pyxlsb import open_workbook

def le_informe(arquivo, aba, titulo):
    """Value of the month in column "Total" of the table titled `titulo` of the sheet. The columns are
    located by the header labels ("Ano", "Mês", "Total"), never by position."""
    with open_workbook(arquivo) as wb:
        with wb.get_sheet(aba) as sh:
            linhas = [[c.v for c in r] for r in sh.rows()]
    ini = next(i for i, r in enumerate(linhas)
               if any(isinstance(v, str) and v.strip().startswith(titulo) for v in r))
    cab = next(i for i in range(ini, len(linhas)) if "Total" in linhas[i] and "Mês" in linhas[i])
    c_ano, c_mes, c_tot = (linhas[cab].index(x) for x in ("Ano", "Mês", "Total"))
    ano_alvo, mes_alvo = int(REFERENCIA[:4]), int(REFERENCIA[5:7])
    ano = None
    for r in linhas[cab + 1:]:
        if len(r) <= c_tot:
            continue
        if isinstance(r[c_ano], (int, float)):
            ano = int(r[c_ano])
        elif isinstance(r[c_ano], str) and r[c_ano].startswith("Total") and ano == ano_alvo:
            break
        if ano == ano_alvo and isinstance(r[c_mes], (int, float)) and int(r[c_mes]) == mes_alvo:
            return float(r[c_tot])
    raise ValueError(f"{aba} / {titulo}: mes {REFERENCIA} nao encontrado")

if ARQ_INFORME.exists():
    informe_kwh = le_informe(ARQ_INFORME, "2.1_Consumo Faturado", "Consumo Faturado (Livres)")
    print(f"energia livres no Informe de Gestao : {informe_kwh / 1e3:,.0f} MWh")
    print(f"energia faturada na base SAMP LV    : {energia_kwh / 1e3:,.0f} MWh")
    print(f"diferenca (Informe - base)          : {(informe_kwh - energia_kwh) / 1e3:,.0f} MWh "
          "(energia faturada fora da base SAMP LV, ex.: suprimento a outra distribuidora)")
else:
    print(f"ATENCAO: Informe de Gestao nao encontrado ({ARQ_INFORME.name}) - conferencia nao feita")

---
## Passo 2 — Separar o que é fornecimento (tarifável × não tarifável)

Cada documento traz várias grandezas em colunas. Para valorizar cada uma pela sua tarifa, o documento é
desdobrado em **linhas**, uma por grandeza com quantidade ou receita (o equivalente aos lançamentos do BW
em SP):

| Coluna de quantidade | Coluna de receita | Grandeza | Posto |
|---|---|---|---|
| `CONS_PONTA_TUSD` | `RECEITA_PONTA` | consumo (kWh) | ponta |
| `CONS_FPONTA_TUSD` | `RECEITA_FPONTA` | consumo (kWh) | fora ponta |
| `CONS_INTERMED_TUSD`, `CONS_RESERVADO_TUSD`, `CONS_NSA_TUSD` | idem | consumo (kWh) | intermediário, reservado, sem posto |
| `DEM_PONTA`, `DEM_FPONTA`, `DEM_NSA` | `RECEITA_DEM_*` | demanda (kW) | ponta, fora ponta, única |

Têm tarifa homologada por item só o consumo e a demanda. O resto tem receita, mas não tem tarifa por
item, e fica marcado como **não tarifável**, linha a linha, para que o total continue igual ao da base:

- ultrapassagem de demanda (`RECEITA_ULT_DEM_*`): é penalidade, não item tarifário (como em SP);
- bandeira (`RECEITA_BANDEIRA`), ICMS, PIS e COFINS;
- **Outros**: a diferença entre o `TOTAL_FAT` do documento e as colunas acima. Na base de livres, ela
  inclui documentos que só trazem o total (pendência com a área: provavelmente demanda contratada de
  clientes desligados).

In [ ]:
# (quantity column, revenue column, grandeza, posto suffix, text)
MEDIDAS_TARIFAVEIS = [
    ("CONS_PONTA_TUSD",     "RECEITA_PONTA",      "CONSUMO", "P",   "Consumo ponta TUSD"),
    ("CONS_FPONTA_TUSD",    "RECEITA_FPONTA",     "CONSUMO", "FP",  "Consumo fora ponta TUSD"),
    ("CONS_INTERMED_TUSD",  "RECEITA_INTERMED",   "CONSUMO", "INT", "Consumo intermediario TUSD"),
    ("CONS_RESERVADO_TUSD", "RECEITA_RESERVADO",  "CONSUMO", "RES", "Consumo horario reservado TUSD"),
    ("CONS_NSA_TUSD",       "RECEITA_NSA",        "CONSUMO", "",    "Consumo sem posto TUSD"),
    ("DEM_PONTA",           "RECEITA_DEM_PONTA",  "DEMANDA", "P",   "Demanda ponta"),
    ("DEM_FPONTA",          "RECEITA_DEM_FPONTA", "DEMANDA", "FP",  "Demanda fora ponta"),
    ("DEM_NSA",             "RECEITA_DEM_NSA",    "DEMANDA", "",    "Demanda unica"),
]
# non-tariff revenue: (quantity column or None, revenue column, text)
MEDIDAS_NAO_TARIFAVEIS = [
    ("ULT_DEM_PONTA",  "RECEITA_ULT_DEM_PONTA",  "Ultrapassagem demanda ponta"),
    ("ULT_DEM_FPONTA", "RECEITA_ULT_DEM_FPONTA", "Ultrapassagem demanda fora ponta"),
    ("ULT_DEM_NSA",    "RECEITA_ULT_DEM_NSA",    "Ultrapassagem demanda unica"),
    (None, "RECEITA_BANDEIRA", "Bandeira tarifaria"),
    (None, "ICMS",             "ICMS"),
    (None, "PIS",              "PIS"),
    (None, "COFINS",           "COFINS"),
    (None, "OUTROS",           "Outros (TOTAL_FAT menos as demais colunas)"),
]

# "Outros": what TOTAL_FAT has beyond the columns above, document by document
_explicado = lv[[m[1] for m in MEDIDAS_TARIFAVEIS] + [m[1] for m in MEDIDAS_NAO_TARIFAVEIS[:-1]]].sum(axis=1)
lv["OUTROS"] = lv["TOTAL_FAT"] - _explicado

ID_DOC = ["LINHA_EXCEL", "Nº doc.", "Cta.contrato", "Código Arquivo", "TIPO_FATURAMENTO",
          "Ctg.tar.", "ZnTemp.", "Data"]
partes = []
for qtd, rec, grandeza, posto, texto in MEDIDAS_TARIFAVEIS:
    m = (lv[qtd] != 0) | (lv[rec] != 0)
    p = lv.loc[m, ID_DOC].copy()
    p["GRP_QUANT"], p["GRP_MONT"], p["GRP_QUANT_TXT"] = qtd, rec, texto
    p["GRANDEZA"], p["SUFIXO_POSTO"], p["QTD"], p["RECEITA_BRL"] = grandeza, posto, lv.loc[m, qtd], lv.loc[m, rec]
    p["SITUACAO"] = "SIM"
    partes.append(p)
for qtd, rec, texto in MEDIDAS_NAO_TARIFAVEIS:
    m = lv[rec].abs() > 1e-9
    p = lv.loc[m, ID_DOC].copy()
    p["GRP_QUANT"], p["GRP_MONT"], p["GRP_QUANT_TXT"] = (qtd or ""), rec, texto
    p["GRANDEZA"] = "DEMANDA" if qtd else "NA"
    p["SUFIXO_POSTO"] = ""
    p["QTD"] = lv.loc[m, qtd] if qtd else 0.0
    p["RECEITA_BRL"] = lv.loc[m, rec]
    p["SITUACAO"] = "NÃO"
    partes.append(p)
linhas = pd.concat(partes, ignore_index=True)

# re-billing: the reversal (negative) and the new bill (positive) are kept apart. Grouped together, their
# quantities cancel while the revenues do not (e.g. a discount removed on re-billing), leaving a group with
# revenue and no quantity to value
_refat = linhas["TIPO_FATURAMENTO"] == "REFAT"
_negativo = (linhas["QTD"] < 0) | ((linhas["QTD"] == 0) & (linhas["RECEITA_BRL"] < 0))
linhas.loc[_refat & _negativo, "TIPO_FATURAMENTO"] = "REFAT estorno"
linhas.loc[_refat & ~_negativo, "TIPO_FATURAMENTO"] = "REFAT conta nova"

# the unfolding keeps every real of the base
d = abs(linhas["RECEITA_BRL"].sum() - lv["TOTAL_FAT"].sum())
print(f"{len(lv)} documentos -> {len(linhas)} linhas "
      f"({(linhas['SITUACAO'] == 'SIM').sum()} tarifaveis, {(linhas['SITUACAO'] == 'NÃO').sum()} nao tarifaveis)")
print(f"soma das linhas x TOTAL_FAT da base: diferenca {d:,.6f}")
assert d < 0.01
print(linhas[linhas["SITUACAO"] == "NÃO"].groupby("GRP_QUANT_TXT")["RECEITA_BRL"].sum()
      .div(1e6).round(2).rename("MR$").to_string())

---
## Passo 3 — Montar a chave tarifária

A tarifa não é um número por cliente: é um número por combinação de características do fornecimento. A
chave é a mesma de SP, a concatenação **sem separador** de oito atributos:

```
SUBGRUPO + MODALIDADE + CLASSE + SUBCLASSE + DETALHE + ACESSANTE + POSTO + UNIDADE
```

Na base SAMP LV, a chave sai da categoria tarifária (`Ctg.tar.`) e da coluna de cada linha:

| Atributo | Origem | Exemplo |
|---|---|---|
| Subgrupo | parte antes do `_` da `Ctg.tar.` | `A4_LVVD` → A4; `A3A_LVAZ` → A3a |
| Modalidade | sufixo da `Ctg.tar.` | `LVVD` → Verde; `LVAZ` → Azul; `GER` → Geração |
| Classe, subclasse, detalhe | o grupo A não diferencia | Não se aplica |
| Acessante | só geradoras A2 e distribuidoras acessantes têm tarifa própria (Passo 4b) | Não se aplica |
| Posto | prefixo E (energia) ou D (demanda) + sufixo do posto | `EFP`, `EP`, `DP`, `D` |
| Unidade | energia ou demanda | MWh, kW |

A classe de consumo (Comercial, Industrial...) não entra na chave, mas identifica o grupo na saída: vem
da subclasse (`ZnTemp.`) pelo de-para `CL_SUB_TAR.xlsx`, o mesmo da RNF de CE e RJ.

**Subclasse `REVENDA` = uso da distribuição.** São outras distribuidoras (permissionárias e
concessionárias) conectadas à rede: a categoria é `A*_LVAZ`, mas a receita é exatamente a tarifa
"Distribuição" do PCAT (por exemplo, A4: 18,12 R$/MWh e 26,34 R$/kW fora ponta). Por isso essas linhas
recebem a modalidade Distribuição, a energia sem posto (o PCAT publica uma tarifa única de energia) e a
classe "Uso da Distribuição" (classe 09 do Informe de Gestão).

In [ ]:
NAO_SE_APLICA = "Não se aplica"
SUBGRUPO_CTG = {"A1": "A1", "A2": "A2", "A3": "A3", "A3A": "A3a", "A4": "A4", "AS": "AS"}
MODALIDADE_CTG = {"LVAZ": "Azul", "LVVD": "Verde", "GER": "Geração"}
FLAG_CTG = {"LVAZ": "Cliente Livre", "LVVD": "Cliente Livre", "GER": "GERAÇÃO"}
SUBCLASSE_DISTRIBUICAO = "REVENDA"        # other distributors using the network (Passo 3 text)

_ctg = linhas["Ctg.tar."].astype(str).str.upper().str.split("_", n=1, expand=True)
linhas["SUBGRUPO"] = _ctg[0].map(SUBGRUPO_CTG)
linhas["MODALIDADE"] = _ctg[1].map(MODALIDADE_CTG)
linhas["FLAG_MERCADO"] = _ctg[1].map(FLAG_CTG)
sem_ctg = linhas["SUBGRUPO"].isna() | linhas["MODALIDADE"].isna()
if sem_ctg.any():
    raise ValueError(f"categoria tarifaria sem de-para: {sorted(linhas.loc[sem_ctg, 'Ctg.tar.'].unique())}")

# network use by other distributors: tariff "Distribuição" of the PCAT
e_dist = linhas["ZnTemp."] == SUBCLASSE_DISTRIBUICAO
linhas.loc[e_dist, "MODALIDADE"] = "Distribuição"
linhas.loc[e_dist, "FLAG_MERCADO"] = "DISTRIBUIÇÃO"
linhas["RECLASSIFICADA"] = e_dist           # inference: subclass REVENDA read as network use (Passo 7b)

linhas["CLASSE"] = NAO_SE_APLICA
linhas["SUBCLASSE"] = NAO_SE_APLICA
linhas["DETALHE"] = NAO_SE_APLICA
linhas["ACESSANTE"] = NAO_SE_APLICA
linhas["UNIDADE"] = np.where(linhas["GRANDEZA"] == "CONSUMO", "MWh", "kW")
linhas["POSTO"] = np.where(linhas["GRANDEZA"] == "CONSUMO", "E", "D") + linhas["SUFIXO_POSTO"]
# generators: a single TUSDg demand, without time-of-use period
linhas.loc[linhas["MODALIDADE"] == "Geração", "POSTO"] = np.where(
    linhas.loc[linhas["MODALIDADE"] == "Geração", "GRANDEZA"] == "DEMANDA", "D", "E")
# distribution: a single energy tariff (no time-of-use period); demand keeps peak / off-peak
linhas.loc[e_dist & (linhas["GRANDEZA"] == "CONSUMO"), "POSTO"] = "E"

# consumption class from the subclass (same de-para of the RNF); "Classe principal" of the output
de_para = pd.read_excel(ARQ_CLASSE, usecols="I:J").dropna()
CLASSE_DA_SUBCLASSE = dict(zip(de_para["Subclasse"], de_para["Classe"]))
CLASSE_DA_SUBCLASSE[SUBCLASSE_DISTRIBUICAO] = "Uso da Distribuição"
linhas["CLASSE_PRINCIPAL"] = linhas["ZnTemp."].map(CLASSE_DA_SUBCLASSE)
sem_classe = linhas["CLASSE_PRINCIPAL"].isna()
if sem_classe.any():
    print(f"ATENCAO: subclasse(s) sem classe no de-para {ARQ_CLASSE.name}: "
          f"{sorted(linhas.loc[sem_classe, 'ZnTemp.'].unique())} - ficam como '(sem de-para)'")
    linhas.loc[sem_classe, "CLASSE_PRINCIPAL"] = "(sem de-para)"

In [ ]:
_long = pd.read_csv(ARQ_TARIFA_LONG, sep=";", decimal=",", encoding="utf-8-sig", dtype={"VIGENCIA": str})
_long = _long[_long["DISTRIBUIDORA"] == DISTRIBUIDORA].copy()

def monta_chave_linhas(df):
    return (df["SUBGRUPO"] + df["MODALIDADE"] + df["CLASSE"] + df["SUBCLASSE"] + df["DETALHE"]
            + df["ACESSANTE"] + df["POSTO"] + df["UNIDADE"])

linhas["CHAVE"] = monta_chave_linhas(linhas)
print("chaves distintas nas linhas tarifaveis:", linhas.loc[linhas["SITUACAO"] == "SIM", "CHAVE"].nunique())

---
## Passo 4 — As tarifas homologadas

As tarifas vêm da tabela comum `common/tbl/tarifas/tarifa_long.csv` (gerada por
`common/3-Py/extrai_pcat.py`): uma linha por distribuidora, vigência, camada e chave. O posto do PCAT,
escrito por extenso, é convertido para o mesmo código da base (E/D + P, FP, INT).

**Tarifa do mês.** A base SAMP LV não traz o mês de referência nem a data de leitura de cada documento
(pendência com a TI). Por isso todas as linhas usam a tarifa do mês contábil: as vigências que tocam o mês
são ponderadas pelos dias do mês em cada uma. Como a ponderação é linear, `TA = BE + CVA + FIN` continua
exata.

In [ ]:
CAMADAS = ["TA", "BE", "CVA", "FIN"]
SUFIXO_POSTO = {"Ponta": "P", "Fora ponta": "FP", "Intermediário": "INT", NAO_SE_APLICA: ""}

def _chave_pcat(row):
    posto = ("D" if row["UNIDADE"] == "kW" else "E") + SUFIXO_POSTO.get(row["POSTO"], "")
    return "".join([row["SUBGRUPO"], row["MODALIDADE"], row["CLASSE"], row["SUBCLASSE"],
                    row["DETALHE"], row["ACESSANTE"], posto, row["UNIDADE"]])

_long["VIGENCIA_INICIO"] = pd.to_datetime(_long["VIGENCIA_INICIO"]).dt.date
_long["VIGENCIA_FIM"] = pd.to_datetime(_long["VIGENCIA_FIM"]).dt.date
_long["CHAVE"] = _long.apply(_chave_pcat, axis=1)

_NAO_COMPONENTE = {"DISTRIBUIDORA", "VIGENCIA", "ANO_VIGENCIA", "VIGENCIA_INICIO", "VIGENCIA_FIM", "CAMADA",
                   "CHAVE", "ORIGEM", "SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE",
                   "POSTO", "UNIDADE", "ACESSANTE", "Total TUSD", "Total TE"}
COMPONENTES = [c for c in _long.columns if c not in _NAO_COMPONENTE]

# weight of each vintage in the month: days of the month inside the vintage
_ini_mes = dt.date(int(REFERENCIA[:4]), int(REFERENCIA[5:7]), 1)
_fim_mes = _ini_mes.replace(day=monthrange(_ini_mes.year, _ini_mes.month)[1])
_dias_mes = (_fim_mes - _ini_mes).days + 1
PESO_MES = {}
for vig, g in _long.groupby("VIGENCIA"):
    ini, fim = max(g["VIGENCIA_INICIO"].iloc[0], _ini_mes), min(g["VIGENCIA_FIM"].iloc[0], _fim_mes)
    if fim >= ini:
        PESO_MES[vig] = ((fim - ini).days + 1) / _dias_mes
if abs(sum(PESO_MES.values()) - 1.0) > 1e-9:
    raise ValueError(f"vigencias nao cobrem o mes {REFERENCIA}: {PESO_MES}")

def monta_tarifa():
    """Month tariff per layer (camada -> DataFrame chave x componente); a key missing in one vintage
    of the month uses the others, with the weight redistributed."""
    tarifa = {}
    for camada in CAMADAS:
        base = _long[_long["CAMADA"] == camada].set_index(["VIGENCIA", "CHAVE"])[COMPONENTES]
        linhas_t, chaves = [], []
        for chave in sorted(set(_long["CHAVE"])):
            disp = [(v, p) for v, p in PESO_MES.items() if (v, chave) in base.index]
            if not disp:
                continue
            tot = sum(p for _, p in disp)
            linhas_t.append(sum((p / tot) * base.loc[(v, chave)].astype(float).values for v, p in disp))
            chaves.append(chave)
        tarifa[camada] = pd.DataFrame(linhas_t, index=chaves, columns=COMPONENTES)
    return tarifa

TARIFA = monta_tarifa()
print("vigencias em", REFERENCIA, ":", ", ".join(f"{v} ({100 * p:.1f}%)" for v, p in PESO_MES.items()))
print(f"{len(COMPONENTES)} componentes | tarifa do mes para {len(TARIFA['TA'])} chaves")
maior = (TARIFA["TA"] - TARIFA["BE"] - TARIFA["CVA"] - TARIFA["FIN"]).abs().max().max()
print(f"TA = BE + CVA + FIN, diferenca maxima: {maior:.10f}")

### Passo 4b — Acessantes com tarifa própria: geradoras A2 e distribuidoras

Algumas tarifas do PCAT são publicadas por acessante: a TUSDg de cada usina do A2 e a tarifa
"Distribuição" de cada distribuidora conectada. A base SAMP LV não traz o nome do acessante, mas traz a
quantidade e a receita. A tarifa aplicada (receita ÷ quantidade) é comparada com a tarifa de cada
acessante do mesmo subgrupo e posto, e a linha recebe o acessante cuja tarifa coincide — cheia ou com o
desconto de fonte incentivada de 50%.

- Quando dois acessantes têm a mesma tarifa (caso comum na Distribuição), o valor da abertura é o mesmo
  com qualquer um deles; a linha recebe o primeiro em ordem alfabética, e o fato é informado.
- Linha sem quantidade (só receita) recebe o acessante já identificado na mesma conta contrato.
- Linha sem coincidência fica sem tarifa (`SEM TARIFA` na saída), para investigação.

In [ ]:
COLS_TUSD = [c for c in COMPONENTES if not c.startswith("TE")]
ACESSANTES = (_long[_long["ACESSANTE"] != NAO_SE_APLICA].groupby(["SUBGRUPO", "MODALIDADE"])["ACESSANTE"]
              .apply(lambda s: sorted(set(s))).to_dict())

alvo = (linhas["SITUACAO"] == "SIM") & linhas.set_index(["SUBGRUPO", "MODALIDADE"]).index.isin(list(ACESSANTES))
acessante_da_conta, avisos = {}, []
linhas["ACESSANTE_INFERIDO"] = False
linhas["ACESSANTES_MESMA_TARIFA"] = ""
ordem = linhas.index[alvo].to_series().sort_values(key=lambda i: linhas.loc[i, "QTD"].eq(0))   # quantity first
for i in ordem:
    r = linhas.loc[i]
    cands = ACESSANTES[(r["SUBGRUPO"], r["MODALIDADE"])]
    chave_de = lambda a: monta_chave_linhas(r.to_frame().T.assign(ACESSANTE=a)).iloc[0]
    validos = [a for a in cands if chave_de(a) in TARIFA["TA"].index]
    conta = (r["Cta.contrato"], r["SUBGRUPO"], r["MODALIDADE"])
    if r["QTD"] == 0:
        escolhido = acessante_da_conta.get(conta, validos[0] if validos else None)
    else:
        fator = 1000.0 if r["GRANDEZA"] == "CONSUMO" else 1.0
        t = r["RECEITA_BRL"] / r["QTD"] * fator
        ok = [a for a in validos if min(abs(TARIFA["TA"].loc[chave_de(a), COLS_TUSD].sum() * k - t)
                                        for k in (1.0, 0.5)) < 0.01]
        escolhido = ok[0] if ok else None
        if len(ok) > 1:
            linhas.at[i, "ACESSANTES_MESMA_TARIFA"] = ", ".join(ok)
            avisos.append(f"{r['MODALIDADE']} {r['SUBGRUPO']}: tarifa {t:.2f} igual em {ok} -> {ok[0]}")
        if not ok:
            avisos.append(f"ATENCAO: {r['MODALIDADE']} {r['SUBGRUPO']}, documento {r['Nº doc.']}: tarifa aplicada "
                          f"{t:.2f} sem acessante correspondente - fica sem tarifa")
    if escolhido:
        linhas.at[i, "ACESSANTE"] = escolhido
        linhas.at[i, "ACESSANTE_INFERIDO"] = True
        acessante_da_conta.setdefault(conta, escolhido)
linhas["CHAVE"] = monta_chave_linhas(linhas)

res = linhas[alvo].groupby(["MODALIDADE", "SUBGRUPO", "ACESSANTE"])["RECEITA_BRL"].agg(["size", "sum"])
res["sum"] = res["sum"] / 1e6
print(res.rename(columns={"size": "linhas", "sum": "MR$"}).round(3).to_string())
for a in sorted(set(avisos)):
    print(a)

### Passo 4c — Classificação dos livres: APE e fonte incentivada

A base não marca o APE (autoprodutor por equiparação) nem o cliente de fonte incentivada, mas o resultado
do faturamento mostra os dois, conta contrato a conta contrato:

- **APE.** A tarifa APE é a TUSD sem os encargos setoriais, um valor que não coincide com nenhum outro do
  PCAT (A4: 89,1 R$/MWh fora ponta, contra 223,6 do livre). A conta é APE quando a tarifa aplicada na
  energia fora ponta coincide com a tarifa APE da chave. O fora ponta é usado porque o desconto de fonte
  incentivada não incide nele (não há fio na energia fora ponta), então não há confusão possível.
- **Fonte incentivada.** O desconto aparece na tarifa de demanda aplicada, abaixo da TUSD cheia da chave,
  e no campo `SUBSCARFI` da base. Os dois sinais são comparados; divergência é informada.

A classificação vai para a coluna `Flag Cli.Liv./ Cat.` (`Cliente Livre APE`, `Cliente Livre Fonte
Incentivada`, `Cliente Livre APE Fonte Incentivada` ou `Cliente Livre`) e para a aba
`APE_Fonte_Incentivada` da saída. É uma **inferência pelo faturamento**, não pelo cadastro: a valorização
continua pela chave da base, e as contas APE seguem na lista de exceções (Passo 7b) até o faturamento
confirmar.

In [ ]:
linhas["FLAG_BASE"] = linhas["FLAG_MERCADO"]        # flag used by the valuation rule (Passo 6)
livre = linhas["FLAG_BASE"] == "Cliente Livre"
_tusd_k = TARIFA["TA"][[c for c in COMPONENTES if not c.startswith("TE")]].sum(axis=1)

def tarifa_aplicada(df, cols_q, cols_r, fator):
    q = df[cols_q].sum(axis=1).groupby(df["Cta.contrato"]).sum()
    r = df[cols_r].sum(axis=1).groupby(df["Cta.contrato"]).sum()
    return (r / q.where(q != 0) * fator)

docs_livres = lv[lv["Cta.contrato"].isin(linhas.loc[livre, "Cta.contrato"])
                 & (lv["ZnTemp."] != SUBCLASSE_DISTRIBUICAO)]
contas = docs_livres.groupby("Cta.contrato").agg(CTG=("Ctg.tar.", "first"), SUBSCARFI=("SUBSCARFI", "sum"))
_ctg = contas["CTG"].str.upper().str.split("_", n=1, expand=True)
contas["SUBGRUPO"], contas["MODALIDADE"] = _ctg[0].map(SUBGRUPO_CTG), _ctg[1].map(MODALIDADE_CTG)
def chave_conta(r, detalhe, posto, unidade):
    return f"{r['SUBGRUPO']}{r['MODALIDADE']}{NAO_SE_APLICA * 2}{detalhe}{NAO_SE_APLICA}{posto}{unidade}"

# APE: off-peak energy tariff of the contract equal to the APE key tariff (and different from the free one)
contas["T_EFP"] = tarifa_aplicada(docs_livres, ["CONS_FPONTA_TUSD"], ["RECEITA_FPONTA"], 1000.0)
contas["T_APE"] = contas.apply(lambda r: _tusd_k.get(chave_conta(r, "APE", "EFP", "MWh"), np.nan), axis=1)
contas["T_LIVRE"] = contas.apply(lambda r: _tusd_k.get(chave_conta(r, NAO_SE_APLICA, "EFP", "MWh"), np.nan), axis=1)
contas["APE"] = ((contas["T_EFP"] - contas["T_APE"]).abs() < 0.05) & ((contas["T_APE"] - contas["T_LIVRE"]).abs() > 0.05)

# incentivized source: demand tariff of the contract below the full key tariff
_dem_q = ["DEM_PONTA", "DEM_FPONTA", "DEM_NSA"]
_dem_r = ["RECEITA_DEM_PONTA", "RECEITA_DEM_FPONTA", "RECEITA_DEM_NSA"]
cheia = []
for cta, r in contas.iterrows():
    d = docs_livres[docs_livres["Cta.contrato"] == cta]
    valor_cheio = 0.0
    for qc, posto in zip(_dem_q, ["DP", "DFP", "D"]):
        # absolute values: a re-billing (reversal + new bill) would otherwise net the demand to zero
        kw = d[qc].abs().sum()
        if kw != 0:                    # Azul has no single demand and Verde no peak / off-peak demand
            valor_cheio += kw * _tusd_k.get(chave_conta(r, NAO_SE_APLICA, posto, "kW"), np.nan)
    cheia.append(valor_cheio)
contas["DEM_CHEIA"] = cheia
contas["DEM_FATURADA"] = docs_livres[_dem_r].abs().groupby(docs_livres["Cta.contrato"]).sum().sum(axis=1)
contas["DESC_DEMANDA"] = 1.0 - contas["DEM_FATURADA"] / contas["DEM_CHEIA"].where(contas["DEM_CHEIA"] > 0)
contas["FONTE_INCENTIVADA"] = contas["DESC_DEMANDA"] > 0.02

contas["CLASSIFICACAO"] = ("Cliente Livre" + np.where(contas["APE"], " APE", "")
                           + np.where(contas["FONTE_INCENTIVADA"], " Fonte Incentivada", ""))
linhas.loc[livre, "FLAG_MERCADO"] = linhas.loc[livre, "Cta.contrato"].map(contas["CLASSIFICACAO"]).fillna("Cliente Livre")

print(contas["CLASSIFICACAO"].value_counts().rename("contas contrato").to_string())
print("\npercentual de desconto na demanda (contas de fonte incentivada):")
print(contas.loc[contas["FONTE_INCENTIVADA"], "DESC_DEMANDA"].round(1).value_counts().sort_index()
      .rename("contas").to_string())
diverge = contas[contas["FONTE_INCENTIVADA"] != (docs_livres["SUBSCARFI"].abs().groupby(docs_livres["Cta.contrato"]).sum() > 0)]
print(f"\nfonte incentivada pela tarifa x SUBSCARFI da base: {len(diverge)} conta(s) divergente(s)"
      + (f": {sorted(diverge.index)}" if len(diverge) else ""))
estranhas = contas[(contas["DESC_DEMANDA"] < -0.02) | contas["DESC_DEMANDA"].between(0.02, 0.45)
                   | contas["DESC_DEMANDA"].between(0.55, 0.95)]
if len(estranhas):
    print(f"ATENCAO: {len(estranhas)} conta(s) com desconto fora de 0%, 50% ou 100% "
          f"(vão para a lista de exceções): {sorted(estranhas.index)}")

# list for the output sheet: one row per document of the classified contracts
APE_FI = (lv.loc[lv["Cta.contrato"].isin(contas.index[contas["APE"] | contas["FONTE_INCENTIVADA"]]),
                 ["Nº doc.", "Cta.contrato"]]
          .assign(**{"Classificação": lambda d: d["Cta.contrato"].map(contas["CLASSIFICACAO"])
                     .str.replace("Cliente Livre ", "", regex=False),
                     "% desconto fonte incentivada": lambda d: d["Cta.contrato"].map(
                         contas["DESC_DEMANDA"].where(contas["FONTE_INCENTIVADA"])).round(2)})
          .sort_values(["Classificação", "Cta.contrato", "Nº doc."]))

---
## Passo 5 — A quantidade

O consumo é convertido de kWh para MWh, porque a tarifa de energia é publicada em R$/MWh; a demanda fica
em kW, como a tarifa de demanda. No livre, a quantidade e a receita vêm na mesma linha, só na TUSD.

In [ ]:
linhas["QUANTIDADE"] = np.where(linhas["GRANDEZA"] == "CONSUMO", linhas["QTD"] / 1000.0, linhas["QTD"])
# columns of the output, as in SP (kWh / kW as billed)
eh_consumo = linhas["GRANDEZA"] == "CONSUMO"
linhas["QTD_CONSUMO_TUSD"] = np.where(eh_consumo, linhas["QTD"], 0.0)
linhas["QTD_CONSUMO_TE"] = 0.0
linhas["DEMANDA_KW"] = np.where(eh_consumo, 0.0, linhas["QTD"])
linhas["TOTAL_CONSUMO_XLS"] = linhas["QTD_CONSUMO_TUSD"]
linhas["TOTAL_DEMANDA_XLS"] = linhas["DEMANDA_KW"]
tar = linhas["SITUACAO"] == "SIM"
print(f"energia tarifavel: {linhas.loc[tar & eh_consumo, 'QUANTIDADE'].sum():,.0f} MWh | "
      f"demanda tarifavel: {linhas.loc[tar & ~eh_consumo, 'QUANTIDADE'].sum() / 1e3:,.0f} MW")

---
## Passo 6 — Quais parcelas incidem

| Mercado | Grandeza | Aplica TUSD | Aplica TE |
|---|---|:-:|:-:|
| Livre | consumo e demanda | sim | não |
| Geração | demanda (TUSDg) | sim | não |
| Distribuição (outras distribuidoras) | consumo e demanda | sim | não |

O livre compra a energia no mercado livre: a distribuidora fatura só o uso da rede (TUSD). A TE fica zero.

Cada item pertence a uma parcela (TUSD ou TE) e a uma família (Fio A, Fio B, Encargos, Perdas, Energia,
RI, Outros) — as mesmas de SP.

In [ ]:
REGRA_VALORIZACAO = {
    ("Cliente Livre", "CONSUMO"): ("TUSD",),
    ("Cliente Livre", "DEMANDA"): ("TUSD",),
    ("GERAÇÃO",       "DEMANDA"): ("TUSD",),
    ("GERAÇÃO",       "CONSUMO"): ("TUSD",),
    ("DISTRIBUIÇÃO",  "CONSUMO"): ("TUSD",),
    ("DISTRIBUIÇÃO",  "DEMANDA"): ("TUSD",),
}

def parcela_do_componente(nome):
    return "TE" if nome.startswith("TE") else "TUSD"

FAMILIA = {
    "FIO A":   ["TUSD_RB", "TUSD_FR", "TUSD_CCT", "TUSD_CCD", "TUSD_CUSD", "TUSDG_T",
                "TE_TRANSPORTE_ITAIPU", "TE_TUST_ITAIPU", "TE_TUST_CI"],
    "FIO B":   ["TUSD_FioB"],
    "ENCARGOS":["TUSD_CDE_COVID", "TUSD_TFSEE", "TUSD_PeD", "TUSD_ONS", "TUSD_CCC", "TUSD_CDE",
                "TUSD_PROINFA", "Liminar1", "TUSDG_ONS", "TE_CDE_COVID", "TE_CDE_ELET",
                "TE_CDE_GD", "TE_PeD", "TE_ESSERR", "TE_CFURH"],
    "PERDAS":  ["TUSD_PT", "TUSD_Per_RB_D", "TUSD_PNT", "TE_Per_RB"],
    "ENERGIA": ["TE_ENERGIA", "TE_ANGRA"],
    "RI":      ["TUSD_RI"],
    "OUTROS":  ["TUSD Subsidio", "TUSD BENEFICIO_L14299", "TUSD Outros",
                "TE Subsidio", "TE BENEFICIO_L14299"],
}
faltando = [c for c in COMPONENTES if c not in {x for l in FAMILIA.values() for x in l}]
assert not faltando, f"componentes sem familia: {faltando}"
print(f"{len(COMPONENTES)} componentes classificados em {len(FAMILIA)} familias")

---
## Passo 7 — Multiplicar: o bloco 1

Para cada linha tarifável, cada um dos 37 itens e cada uma das 4 camadas:

```
valor do item = quantidade × tarifa do item     (zero se a parcela do item não incide na linha)
```

Linha cuja chave não existe na tabela de tarifas não é valorizada: vai para o fim da saída como
`SEM TARIFA`, com a receita, para investigação.

In [ ]:
tem_tarifa = linhas["CHAVE"].isin(TARIFA["TA"].index)
linhas.loc[(linhas["SITUACAO"] == "SIM") & ~tem_tarifa, "SITUACAO"] = "SEM TARIFA"
if (linhas["SITUACAO"] == "SEM TARIFA").any():
    st = linhas[linhas["SITUACAO"] == "SEM TARIFA"]
    print(f"ATENCAO: {len(st)} linha(s) com chave sem tarifa (R$ {st['RECEITA_BRL'].sum():,.2f}):")
    print(st.groupby("CHAVE")["RECEITA_BRL"].agg(["size", "sum"]).to_string())

tarifaveis = linhas[linhas["SITUACAO"] == "SIM"].copy()
parcelas_aplicadas = tarifaveis.apply(lambda r: REGRA_VALORIZACAO.get((r["FLAG_BASE"], r["GRANDEZA"]), ()), axis=1)
aplica = pd.DataFrame({c: parcelas_aplicadas.apply(lambda p, c=c: parcela_do_componente(c) in p)
                       for c in COMPONENTES}, index=tarifaveis.index)

BLOCO1 = {}
for camada in CAMADAS:
    t = TARIFA[camada].reindex(tarifaveis["CHAVE"].values)
    t.index = tarifaveis.index
    BLOCO1[camada] = t[COMPONENTES].mul(tarifaveis["QUANTIDADE"], axis=0).where(aplica, 0.0)
tarifaveis["CALCULADO_TA"] = BLOCO1["TA"].sum(axis=1)
print(f"valor calculado pela tarifa de aplicacao: {tarifaveis['CALCULADO_TA'].sum() / 1e6:,.2f} MR$")
print(f"receita registrada nessas linhas        : {tarifaveis['RECEITA_BRL'].sum() / 1e6:,.2f} MR$")

### Passo 7b — Associação direta da tarifa: conferência e lista de exceções

Uma linha tem **associação direta** quando a chave sai só da base (categoria tarifária + coluna) e a
tarifa aplicada pelo faturamento é compatível com a tarifa dessa chave. Compatível quer dizer: a
receita ÷ quantidade fica entre a tarifa cheia e a tarifa com 100% de desconto no fio (o desconto de
fonte incentivada só incide no Fio A e no Fio B).

Toda linha fora disso é listada, com o motivo, num arquivo próprio de cada mês
(`2-Checks/lista_excecoes/lista_excecoes_<UF>_<ref>.xlsx`), para o faturamento indicar a
tarifa correta de cada conta contrato:

| Motivo | O que aconteceu | O processo segue com |
|---|---|---|
| `CHAVE SEM TARIFA` | a chave montada não existe no PCAT | a linha fora da conta (`SEM TARIFA`) |
| `RECLASSIFICADA COMO DISTRIBUIÇÃO` | subclasse `REVENDA` lida como uso da distribuição | inferência |
| `ACESSANTE INFERIDO PELA TARIFA` | geradora A2 ou distribuidora: acessante escolhido pela tarifa aplicada | inferência |
| `TARIFA APLICADA FORA DA FAIXA DA CHAVE` | a receita não é explicada pela tarifa da chave (ex.: APE) | a chave da base; a diferença vai para a 2ª passada |
| `RECEITA SEM QUANTIDADE` | não há como conferir a tarifa | resíduo não ajustável |
| `DESCONTO FORA DE 0%, 50% OU 100%` | desconto de fonte incentivada atípico na demanda (Passo 4c) | a chave da base |
| `DESCONTO DIFERENTE DO SUBSCARFI` | o desconto pela tarifa (valor cheio − receita) difere do informado na base, num documento da conta, acima de `TOL_DESCONTO_PCT` do desconto calculado e de `TOL_DESCONTO_MIN_AT_BRL` | a chave da base |

As abas `Geradoras` e `APE` listam as contas que o faturamento precisa identificar: o nome da usina de
cada geradora e a confirmação de cada conta classificada como APE pela tarifa (Passo 4c). As colunas a
preencher também têm fundo cinza claro.

A aba `Divergencia_desconto` mostra, para essas contas, todos os documentos (com o tipo de fatura,
porque o refaturamento só se lê com o estorno e a conta nova juntos), o valor total, o desconto pela
tarifa, o desconto informado e a diferença.

Na aba `Por_conta`, cada valor usado pelo processo (`Ctg.tar.`, `ZnTemp.`, `Detalhe`, `Acessante`) vem
seguido de uma coluna com o mesmo nome e o sufixo `correto`, com fundo cinza claro: é ali que o
faturamento preenche o valor certo (em branco = o valor usado está correto).

A coluna `Chave compatível` sugere, quando existe, outra chave do PCAT do mesmo subgrupo e posto cuja
faixa explica a tarifa aplicada (por exemplo, a tarifa APE). É só uma pista para o faturamento: o
processo não troca a chave sozinho.

In [ ]:
COLS_TUSD_FIO = [c for c in FAMILIA["FIO A"] + FAMILIA["FIO B"] if c in COMPONENTES and not c.startswith("TE")]
COLS_TUSD = [c for c in COMPONENTES if not c.startswith("TE")]
_tusd = TARIFA["TA"][COLS_TUSD].sum(axis=1)
_fio = TARIFA["TA"][COLS_TUSD_FIO].sum(axis=1)

aval = linhas[linhas["SITUACAO"].isin(["SIM", "SEM TARIFA"])].copy()
fator = np.where(aval["GRANDEZA"] == "CONSUMO", 1000.0, 1.0)
aval["TARIFA_APLICADA"] = np.where(aval["QTD"] != 0, aval["RECEITA_BRL"] / aval["QTD"].replace(0, np.nan) * fator, np.nan)
aval["TARIFA_CHEIA"] = aval["CHAVE"].map(_tusd)
aval["TARIFA_MINIMA"] = aval["CHAVE"].map(_tusd - _fio)
tol = 0.01 + 0.005 * aval["TARIFA_CHEIA"].abs().fillna(0)
na_faixa = ((aval["TARIFA_APLICADA"] >= aval["TARIFA_MINIMA"] - tol)
            & (aval["TARIFA_APLICADA"] <= aval["TARIFA_CHEIA"] + tol))

motivos = pd.Series([[] for _ in range(len(aval))], index=aval.index)
def marca(mask, texto):
    for i in aval.index[mask]:
        motivos[i].append(texto)
marca(aval["SITUACAO"] == "SEM TARIFA", "CHAVE SEM TARIFA")
marca(aval["RECLASSIFICADA"], "RECLASSIFICADA COMO DISTRIBUIÇÃO")
marca(aval["ACESSANTE_INFERIDO"], "ACESSANTE INFERIDO PELA TARIFA")
marca((aval["SITUACAO"] == "SIM") & (aval["QTD"] != 0) & ~na_faixa, "TARIFA APLICADA FORA DA FAIXA DA CHAVE")
marca((aval["QTD"] == 0) & (aval["RECEITA_BRL"] != 0), "RECEITA SEM QUANTIDADE")
marca(aval["Cta.contrato"].isin(estranhas.index), "DESCONTO FORA DE 0%, 50% OU 100%")

# document-level check of the incentivized-source discount: full-tariff value of the document's tariff
# lines minus the billed revenue, against SUBSCARFI. Free customers only, APE contracts excluded (their
# lower tariff is not a discount).
_liv = aval[(aval["FLAG_BASE"] == "Cliente Livre") & (aval["SITUACAO"] == "SIM")
            & ~aval["Cta.contrato"].isin(contas.index[contas["APE"]])]
_cheio = _liv["QTD"] / np.where(_liv["GRANDEZA"] == "CONSUMO", 1000.0, 1.0) * _liv["TARIFA_CHEIA"]
desc_doc = (_cheio - _liv["RECEITA_BRL"]).groupby(_liv["Nº doc."]).sum()
_docs = lv.set_index("Nº doc.").loc[desc_doc.index]
div = pd.DataFrame({"Nº doc.": desc_doc.index, "Cta.contrato": _docs["Cta.contrato"].values,
                    "Tipo de faturamento": _docs["TIPO_FATURAMENTO"].values,
                    "Código Arquivo": _docs["Código Arquivo"].values, "Data": _docs["Data"].values,
                    "Ctg.tar.": _docs["Ctg.tar."].values, "ZnTemp.": _docs["ZnTemp."].values,
                    "Valor total (TOTAL_FAT, R$)": _docs["TOTAL_FAT"].values,
                    "Desconto pela tarifa (R$)": desc_doc.values,
                    "Desconto informado (SUBSCARFI, R$)": _docs["SUBSCARFI"].values})
div["Diferença (R$)"] = div["Desconto pela tarifa (R$)"] - div["Desconto informado (SUBSCARFI, R$)"]
# a contract enters the list when one of its documents diverges; all its documents are shown, because a
# re-billing (reversal + new bill) is only readable with every document of the contract
_dif = div["Diferença (R$)"].abs()
_base = div["Desconto pela tarifa (R$)"].abs()
# relative test; a zero calculated discount with an informed one has no ratio and passes on the minimum alone
_rel = (_base == 0) | (_dif > TOL_DESCONTO_PCT * _base)
contas_div = set(div.loc[_rel & (_dif > TOL_DESCONTO_MIN_AT_BRL), "Cta.contrato"]) | set(diverge.index)
divergencias = div[div["Cta.contrato"].isin(contas_div)].sort_values(["Cta.contrato", "Data", "Nº doc."])
marca(aval["Cta.contrato"].isin(contas_div), "DESCONTO DIFERENTE DO SUBSCARFI")
print(f"desconto de fonte incentivada x SUBSCARFI, documento a documento: {len(contas_div)} conta(s) com "
      f"diferenca acima de {100 * TOL_DESCONTO_PCT:.1f}% do desconto calculado e de R$ "
      f"{TOL_DESCONTO_MIN_AT_BRL:,.2f} | diferenca liquida "
      f"R$ {divergencias['Diferença (R$)'].sum():,.2f}")
aval["MOTIVO"] = motivos.apply(" | ".join)
excecoes = aval[aval["MOTIVO"] != ""].copy()

# hint: another PCAT key of the same subgroup, unit and period whose range explains the applied tariff
_pcat = _long[_long["CAMADA"] == "TA"][["SUBGRUPO", "MODALIDADE", "DETALHE", "ACESSANTE", "CHAVE"]].drop_duplicates("CHAVE")
_pcat = _pcat[_pcat["CHAVE"].isin(TARIFA["TA"].index)]
def chave_compativel(r):
    if pd.isna(r["TARIFA_APLICADA"]) or "FORA DA FAIXA" not in r["MOTIVO"]:
        return ""
    sufixo = r["POSTO"] + r["UNIDADE"]
    cands = _pcat[(_pcat["SUBGRUPO"] == r["SUBGRUPO"]) & _pcat["CHAVE"].str.endswith(sufixo)
                  & (_pcat["CHAVE"] != r["CHAVE"])]
    ok = [f"{c.MODALIDADE} / {c.DETALHE} / {c.ACESSANTE}" for c in cands.itertuples()
          if _tusd[c.CHAVE] - _fio[c.CHAVE] - 0.02 <= r["TARIFA_APLICADA"] <= _tusd[c.CHAVE] + 0.02]
    return " ; ".join(ok)
excecoes["CHAVE_COMPATIVEL"] = excecoes.apply(chave_compativel, axis=1)

COLS_EXC = {"Cta.contrato": "Cta.contrato", "Nº doc.": "Nº doc.", "LINHA_EXCEL": "Linha no Excel da base",
            "Ctg.tar.": "Ctg.tar.", "ZnTemp.": "ZnTemp.", "TIPO_FATURAMENTO": "Tipo de faturamento",
            "GRP_QUANT_TXT": "Grandeza", "QTD": "Quantidade (kWh ou kW)", "RECEITA_BRL": "Receita (R$)",
            "TARIFA_APLICADA": "Tarifa aplicada (R$/MWh ou R$/kW)", "CHAVE": "Chave usada",
            "TARIFA_CHEIA": "TUSD cheia da chave", "TARIFA_MINIMA": "TUSD da chave com 100% de desconto no fio",
            "ACESSANTE": "Acessante usado", "ACESSANTES_MESMA_TARIFA": "Acessantes com a mesma tarifa",
            "MOTIVO": "Motivo", "CHAVE_COMPATIVEL": "Chave compatível (pista)"}
detalhe = excecoes[list(COLS_EXC)].rename(columns=COLS_EXC).sort_values(["Motivo", "Cta.contrato", "Nº doc."])

# one row per contract: what the billing team is asked to fill in. Each value the process used is
# followed by a column with the same name and the suffix " correto", to be filled by the billing team
# (light grey background); blank means the value used is right.
_txt = lambda s: " | ".join(sorted(set(map(str, s))))
por_conta = excecoes.groupby("Cta.contrato").agg(
    **{"Ctg.tar.": ("Ctg.tar.", _txt),
       "ZnTemp.": ("ZnTemp.", _txt),
       "Detalhe": ("DETALHE", _txt),
       "Acessante": ("ACESSANTE", _txt),
       "Motivo": ("MOTIVO", lambda s: " | ".join(sorted({m for x in s for m in x.split(" | ")}))),
       "Chave compatível (pista)": ("CHAVE_COMPATIVEL", lambda s: " | ".join(sorted(set(s) - {""}))),
       "Documentos": ("Nº doc.", "nunique"),
       "Receita (R$)": ("RECEITA_BRL", "sum")}).reset_index()
A_PREENCHER = []
for col in ["Acessante", "Detalhe", "ZnTemp.", "Ctg.tar."]:            # inserted right after each column
    novo = f"{col} correto"
    por_conta.insert(por_conta.columns.get_loc(col) + 1, novo, "")
    A_PREENCHER.append(novo)
por_conta["Observação do faturamento"] = ""
A_PREENCHER.append("Observação do faturamento")
por_conta = por_conta.sort_values("Receita (R$)", key=abs, ascending=False)

print(f"linhas sem associacao direta: {len(excecoes)} de {len(aval)} | contas contrato: {len(por_conta)} | "
      f"receita {excecoes['RECEITA_BRL'].sum() / 1e6:,.2f} MR$")
resumo = excecoes.assign(M=excecoes["MOTIVO"].str.split(r" \| ")).explode("M").groupby("M").agg(
    linhas=("Nº doc.", "size"), contas=("Cta.contrato", "nunique"), receita_MR=("RECEITA_BRL", lambda s: s.sum() / 1e6))
print(resumo.round(3).to_string())

# contracts the billing team is asked to identify: generating plants (plant name) and APE (confirmation)
def lista_contas(mascara, preencher):
    d = lv[mascara]
    t = d.groupby("Cta.contrato").agg(
        **{"Ctg.tar.": ("Ctg.tar.", _txt), "ZnTemp.": ("ZnTemp.", _txt),
           "Tipos de faturamento": ("TIPO_FATURAMENTO", _txt), "Documentos": ("Nº doc.", "nunique"),
           "Valor total (TOTAL_FAT, R$)": ("TOTAL_FAT", "sum"),
           "SUBSGERFI informado (R$)": ("SUBSGERFI", "sum")}).reset_index()
    for col in preencher:
        t[col] = ""
    return t.sort_values(["Ctg.tar.", "Cta.contrato"])
geradoras = lista_contas(lv["Ctg.tar."].astype(str).str.endswith("_GER"), ["Usina", "Observação do faturamento"])
ape = lista_contas(lv["Cta.contrato"].isin(contas.index[contas["APE"]]),
                   ["Confirma APE? (sim/não)", "Observação do faturamento"]).drop(columns="SUBSGERFI informado (R$)")
A_PREENCHER += ["Usina", "Confirma APE? (sim/não)"]

from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
SAIDA_EXCECOES.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(SAIDA_EXCECOES, engine="openpyxl") as escritor:
    por_conta.to_excel(escritor, sheet_name="Por_conta", index=False)
    detalhe.to_excel(escritor, sheet_name="Detalhe_linhas", index=False)
    # the divergence sheet starts on row 4: rows 1 to 3 carry the tolerance disclaimer
    divergencias.to_excel(escritor, sheet_name="Divergencia_desconto", index=False, startrow=3)
    geradoras.to_excel(escritor, sheet_name="Geradoras", index=False)
    ape.to_excel(escritor, sheet_name="APE", index=False)
    for nome, lin_cab in (("Por_conta", 1), ("Detalhe_linhas", 1), ("Divergencia_desconto", 4),
                          ("Geradoras", 1), ("APE", 1)):
        ws = escritor.book[nome]
        for celula in ws[lin_cab]:
            celula.font = Font(bold=True, size=9, color="FFFFFF")
            celula.fill = PatternFill("solid", fgColor="1F4E79")
            celula.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
        ws.row_dimensions[lin_cab].height = 42
        for col in range(1, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(col)].width = 20
        ws.auto_filter.ref = f"A{lin_cab}:{get_column_letter(ws.max_column)}{ws.max_row}"
    # columns to be filled by the billing team: light grey background
    cinza = PatternFill("solid", fgColor="D9D9D9")
    for nome in ("Por_conta", "Geradoras", "APE"):
        ws = escritor.book[nome]
        for col in range(1, ws.max_column + 1):
            if ws.cell(row=1, column=col).value in A_PREENCHER:
                for linha in range(2, ws.max_row + 1):
                    ws.cell(row=linha, column=col).fill = cinza
            if "(R$)" in str(ws.cell(row=1, column=col).value):
                for linha in range(2, ws.max_row + 1):
                    ws.cell(row=linha, column=col).number_format = "#,##0.00"
    ws = escritor.book["Divergencia_desconto"]
    for col in range(8, ws.max_column + 1):
        for linha in range(5, ws.max_row + 1):
            ws.cell(row=linha, column=col).number_format = "#,##0.00"
    _br = lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")   # 1.234,56
    _pct = f"{100 * TOL_DESCONTO_PCT:.1f}".replace(".", ",")
    _aviso = [
        f"Tolerância: são listadas as contas contrato com ao menos um documento cuja diferença entre o desconto "
        f"pela tarifa e o desconto informado (SUBSCARFI) supera, em valor absoluto, {_pct}% do desconto calculado "
        f"e também R$ {_br(TOL_DESCONTO_MIN_AT_BRL)}.",
        "Documento sem desconto pela tarifa e com desconto informado entra se superar o valor mínimo. "
        "Diferenças menores não são listadas.",
        "Todos os documentos da conta aparecem, para permitir a leitura de refaturamentos (estorno e conta nova).",
    ]
    for _lin, _txt in enumerate(_aviso, start=1):
        ws.cell(row=_lin, column=1, value=_txt).font = Font(bold=True, size=10, color="C00000")
print(f"gravado: {SAIDA_EXCECOES}")

---
## Passo 8 — Agrupar

A saída traz uma linha por **grupo**, não por documento. O grupo junta as linhas que descrevem o mesmo
fornecimento, com as mesmas chaves de SP:

```
Classe principal + Categoria tarifa + Operação principal + Flag mercado + Classe/Sub-classe + Chave
+ Tipo de faturamento
```

A classificação APE / fonte incentivada (Passo 4c) entra na `Flag Cli.Liv./ Cat.` e, portanto, separa os
grupos. Na base SAMP LV: *Operação principal* = `Código Arquivo` (05 ou 06); *Classe/Sub-classe* = subclasse
(`ZnTemp.`); *Grp.estatísts.quant.* e *mont.* = nome das colunas de quantidade e receita. A coluna
`Qtd. Cta.contrato` informa quantas contas contrato formam cada grupo, contando todas as ocorrências
(uma conta com dois documentos no grupo conta duas vezes), para que a soma bata com as listas da base. Em
SP, essa posição trazia as linhas de origem do BW.

Não há faixas de benefício social nos livres; as colunas da Tarifa Social ficam zeradas.

In [ ]:
tarifaveis["CATEGORIA_TARIFA"] = tarifaveis["Ctg.tar."]
tarifaveis["OPERACAO"] = tarifaveis["Código Arquivo"]
tarifaveis["OPERACAO_TXT"] = "SAMP LV - " + tarifaveis["TIPO_FATURAMENTO"]
tarifaveis["SUBOPERACAO"] = ""
tarifaveis["SUBOPERACAO_TXT"] = ""
tarifaveis["CLASSE_SUBCLASSE"] = tarifaveis["ZnTemp."]

CHAVES_DO_GRUPO = ["CLASSE_PRINCIPAL", "CATEGORIA_TARIFA", "OPERACAO", "OPERACAO_TXT",
                   "FLAG_MERCADO", "CLASSE_SUBCLASSE", "CHAVE", "TIPO_FATURAMENTO"]
CONCATENADAS = ["GRP_MONT", "GRP_QUANT", "GRP_QUANT_TXT", "SUBOPERACAO", "SUBOPERACAO_TXT"]

def junta(serie, limite=6):
    """Distinct values, in order of appearance, separated by a bar."""
    vistos, saida = set(), []
    for v in serie:
        t = str(v).strip()
        if t and t.lower() != "nan" and t not in vistos:
            vistos.add(t); saida.append(t)
    if len(saida) > limite:
        return " | ".join(saida[:limite]) + f" | ... (+{len(saida) - limite})"
    return " | ".join(saida)

def lista_linhas(serie, limite=10):
    """Excel rows of the base that make up the group."""
    numeros = sorted({int(v) for v in serie if pd.notna(v)})
    if len(numeros) > limite:
        return "-".join(str(n) for n in numeros[:limite]) + f"-... (+{len(numeros) - limite})"
    return "-".join(str(n) for n in numeros)

grupo = tarifaveis.groupby(CHAVES_DO_GRUPO, dropna=False, sort=False)
ident = grupo.agg(**{c: (c, junta) for c in CONCATENADAS})
ident["LINHAS_ORIGEM"] = grupo["Cta.contrato"].count()
ident["CONSUMO_TUSD"] = grupo["QTD_CONSUMO_TUSD"].sum()
ident["CONSUMO_TE"] = grupo["QTD_CONSUMO_TE"].sum()
ident["DEMANDA"] = grupo["DEMANDA_KW"].sum()
ident["TOTAL_CONSUMO"] = grupo["TOTAL_CONSUMO_XLS"].sum()
ident["TOTAL_DEMANDA"] = grupo["TOTAL_DEMANDA_XLS"].sum()
ident["RECEITA_BRL"] = grupo["RECEITA_BRL"].sum()
ident["SUBSCARFI"] = 0.0          # filled below: discount reported by the base, per group
ident = ident.reset_index()

codigo_grupo = grupo.ngroup()
# energy on which each parcel is due (free market: TUSD only)
_kwh = tarifaveis["TOTAL_CONSUMO_XLS"].where(tarifaveis["GRANDEZA"] == "CONSUMO", 0.0)
for _parcela in ("TUSD", "TE"):
    _devida = parcelas_aplicadas.apply(lambda p, x=_parcela: x in p)
    ident[f"ENERGIA_COM_{_parcela}"] = _kwh.where(_devida, 0.0).groupby(codigo_grupo).sum().values
B1_GRUPO = {c: BLOCO1[c].groupby(codigo_grupo).sum().reset_index(drop=True) for c in CAMADAS}
ident["CALCULADO_TA"] = B1_GRUPO["TA"].sum(axis=1).values

# no social benefit in the free market: Tarifa Social and band columns stay empty, as in SP's free groups
TS_GRUPO = {c: pd.DataFrame(0.0, index=ident.index, columns=COMPONENTES) for c in CAMADAS}
ident["AJUSTE_TARIFA_SOCIAL_BRL"] = 0.0
ident["FAIXA1_PCT"] = np.nan
print(f"{len(tarifaveis)} linhas tarifaveis -> {len(ident)} grupos")

---
## Passo 9 — O ajuste: o bloco 2

A diferença entre o bloco 1 e a receita é corrigida como em SP:

**1. Regime APE.** A base SAMP LV não identifica o autoprodutor por equiparação: a etapa fica zerada
(colunas mantidas, para a saída ser igual à de SP).

**2. 1ª passada — desconto de fonte incentivada.** Clientes livres do grupo A e geradoras (regime `fio`).
O que separa o bloco 1 da receita é absorvido **somente por Fio A e Fio B**, com desconto limitado a
[0%, 100%]. Encargos e perdas ficam intactos (PRORET Submódulo 7.1, §38: o desconto incide sobre a TUSD
Transporte).

**3. 2ª passada — fechamento.** O que sobra é distribuído proporcionalmente entre todos os itens do grupo,
até o total igualar a receita. O tamanho da 2ª passada é o indicador de qualidade do processo.

**Conferência nova em CE e RJ.** A base informa o desconto de fonte incentivada concedido em cada
documento (`SUBSCARFI`). Em SP o desconto só podia ser estimado; aqui o desconto da 1ª passada é
comparado com o informado.

In [ ]:
COLS_FIO = [c for c in FAMILIA["FIO A"] + FAMILIA["FIO B"] if c in COMPONENTES]

ident["REGIME"] = np.where((ident["FLAG_MERCADO"].str.startswith("Cliente Livre") | (ident["FLAG_MERCADO"] == "GERAÇÃO"))
                           & ident["CHAVE"].str.startswith("A"), "fio", "uniforme")
em_fio = ident["REGIME"] == "fio"

# APE: not identified in the SAMP LV base -> step kept, at zero
APE_RED = {c: pd.DataFrame(0.0, index=ident.index, columns=COMPONENTES) for c in CAMADAS}
ident["RECEITA_APE_BRL"] = 0.0
ident["AJUSTE_APE_BRL"] = 0.0

# --- first pass: the gap is absorbed by Fio A + Fio B alone (incentivized-source discount)
fio_ta = B1_GRUPO["TA"][COLS_FIO].sum(axis=1)
falta = ident["CALCULADO_TA"] + ident["AJUSTE_TARIFA_SOCIAL_BRL"] + ident["AJUSTE_APE_BRL"] - ident["RECEITA_BRL"]
desc = np.where(em_fio & (fio_ta != 0), falta / fio_ta.replace(0, np.nan), 0.0)
desc = np.clip(np.nan_to_num(desc), 0.0, 1.0)
ident["DESC_FIO"] = desc

B2_GRUPO = {}
for camada in CAMADAS:
    m = B1_GRUPO[camada] - TS_GRUPO[camada] - APE_RED[camada]
    m[COLS_FIO] = m[COLS_FIO].mul(1.0 - desc, axis=0)
    B2_GRUPO[camada] = m
parcial = B2_GRUPO["TA"].sum(axis=1)
ident["AJUSTE_1A_PASSADA_BRL"] = (parcial - ident["CALCULADO_TA"] - ident["AJUSTE_TARIFA_SOCIAL_BRL"]
                                  - ident["AJUSTE_APE_BRL"])

# --- second pass: what is left is spread over every item of the group
sem_ajuste = parcial == 0
fator2 = np.nan_to_num(np.where(~sem_ajuste, ident["RECEITA_BRL"] / parcial.replace(0, np.nan), 0.0))
if not APLICAR_2A_PASSADA:
    fator2 = np.ones(len(ident))
B2_GRUPO = {c: B2_GRUPO[c].mul(fator2, axis=0) for c in CAMADAS}
ident["PCT_SEGUNDA_PASSADA"] = np.where(sem_ajuste, np.nan, fator2 - 1.0)
ident["AJUSTE_2A_PASSADA_BRL"] = B2_GRUPO["TA"].sum(axis=1) - parcial
ident["DIF_NAO_AJUSTADA_BRL"] = ident["RECEITA_BRL"] - B2_GRUPO["TA"].sum(axis=1)

ident["DIFERENCA_BRL"] = ident["RECEITA_BRL"] - ident["CALCULADO_TA"]
ident["FATOR_AJUSTE"] = np.where(ident["CALCULADO_TA"] != 0, ident["RECEITA_BRL"] / ident["CALCULADO_TA"], np.nan)
ident["PCT_AJUSTE"] = ident["FATOR_AJUSTE"] - 1.0

d = (ident["AJUSTE_TARIFA_SOCIAL_BRL"] + ident["AJUSTE_APE_BRL"] + ident["AJUSTE_1A_PASSADA_BRL"]
     + ident["AJUSTE_2A_PASSADA_BRL"] - ident["DIFERENCA_BRL"])
if APLICAR_2A_PASSADA:
    print(f"1a passada + 2a passada = diferenca total, grupos ajustaveis, diferenca maxima: "
          f"{d[~sem_ajuste].abs().max():.10f}")
print(f"grupos: {len(ident)} | regime fio: {int(em_fio.sum())} | regime uniforme: {int((~em_fio).sum())}")
print(f"sem ajuste possivel (bloco 1 zerado): {int(sem_ajuste.sum())}")
print("\nresumo do bloco 2 (MR$):")
print(f"  1a passada (desconto de fonte incentivada) ... {ident['AJUSTE_1A_PASSADA_BRL'].sum() / 1e6:,.2f}")
print(f"  2a passada (fechamento de conta) ............. {ident['AJUSTE_2A_PASSADA_BRL'].sum() / 1e6:,.2f}")

fio_2a = ident.loc[em_fio, "AJUSTE_2A_PASSADA_BRL"]
residuo = fio_2a[fio_2a.abs() > 0.01]
if APLICAR_2A_PASSADA and len(residuo):
    print(f"\nATENCAO: {len(residuo)} grupo(s) do regime fio precisaram de 2a passada, total "
          f"{residuo.sum() / 1e6:,.4f} MR$ (maiores abaixo)")
    print(ident.loc[residuo.abs().nlargest(5).index, ["CATEGORIA_TARIFA", "GRP_QUANT", "CLASSE_SUBCLASSE",
          "TIPO_FATURAMENTO", "RECEITA_BRL", "CALCULADO_TA", "DESC_FIO", "AJUSTE_2A_PASSADA_BRL"]].to_string())

In [ ]:
# Check: 1st-pass discount x incentivized-source discount reported by the base (SUBSCARFI).
# SUBSCARFI is per document; it is compared with the 1st pass of the documents' groups, in total and by
# tariff category.
_docs_tar = tarifaveis.drop_duplicates("Nº doc.")[["Nº doc.", "Ctg.tar."]]
_subs = lv.set_index("Nº doc.").loc[_docs_tar["Nº doc."], "SUBSCARFI"]
_subs_ctg = _subs.groupby(_docs_tar.set_index("Nº doc.")["Ctg.tar."]).sum()
_desc_ctg = -ident.groupby("CATEGORIA_TARIFA")["AJUSTE_1A_PASSADA_BRL"].sum()
cmp_ = pd.DataFrame({"1a passada (MR$)": _desc_ctg / 1e6, "SUBSCARFI (MR$)": _subs_ctg / 1e6}).fillna(0.0)
cmp_["diferenca (MR$)"] = cmp_["1a passada (MR$)"] - cmp_["SUBSCARFI (MR$)"]
cmp_.loc["Total"] = cmp_.sum()
print("desconto de fonte incentivada: calculado (1a passada) x informado na base (SUBSCARFI)")
print(cmp_.round(3).to_string())
fora_base = lv.loc[~lv["Nº doc."].isin(_docs_tar["Nº doc."]), "SUBSCARFI"].sum()
if abs(fora_base) > 0.01:
    print(f"SUBSCARFI em documentos sem linha tarifavel: {fora_base / 1e6:,.3f} MR$")

---
## Passo 10 — Conferências

| Conferência | O que garante |
|---|---|
| 1. Bloco 2 = receita, grupo a grupo | A abertura fecha no faturado |
| 2. TA = BE + CVA + FIN no bloco 2 | As camadas continuam coerentes depois do ajuste |
| 3. Tarifável + não tarifável = `TOTAL_FAT` da base | Nenhum real se perdeu no desdobramento |

In [ ]:
ok = True
soma_b2 = B2_GRUPO["TA"].sum(axis=1)
d1 = (soma_b2 - ident["RECEITA_BRL"]).where(~sem_ajuste, 0.0).abs().max() if APLICAR_2A_PASSADA else \
     (soma_b2 + ident["DIF_NAO_AJUSTADA_BRL"] - ident["RECEITA_BRL"]).abs().max()
print(f"1. bloco 2 x receita, grupo a grupo ........ diferenca maxima {d1:,.6f}")
ok &= d1 < 0.01
d2 = (B2_GRUPO["TA"] - B2_GRUPO["BE"] - B2_GRUPO["CVA"] - B2_GRUPO["FIN"]).abs().max().max()
print(f"2. TA = BE + CVA + FIN no bloco 2 .......... diferenca maxima {d2:,.6f}")
ok &= d2 < 0.01
fora = linhas[linhas["SITUACAO"] != "SIM"].copy()
d3 = abs(ident["RECEITA_BRL"].sum() + fora["RECEITA_BRL"].sum() - lv["TOTAL_FAT"].sum())
print(f"3. tarifavel + nao tarifavel x TOTAL_FAT ... diferenca {d3:,.6f}")
ok &= d3 < 0.01
if sem_ajuste.any():
    print(f"   residuo nao ajustavel (grupos com bloco 1 zerado): "
          f"{ident.loc[sem_ajuste, 'RECEITA_BRL'].sum() / 1e6:,.4f} MR$ em {int(sem_ajuste.sum())} grupo(s)")
print("\nTODAS AS CONFERENCIAS OK" if ok else "\nATENCAO: CONFERENCIA FALHOU")
assert ok

---
# Parte II — Clientes cativos (base SAMP de documentos)

A construir quando a base completa chegar (Passos 1 a 10 do cativo). As saídas abaixo já recebem só a
Parte I.

---
## Passo 15 — Gerar a planilha

A saída `1-Entrega/pxq/pxq_<UF>_<ref>.xlsx` tem as mesmas abas e colunas da de SP:

| Aba | Conteúdo |
|---|---|
| **Leia-me** | O que cada aba contém e onde é usada |
| **Componentes** | Os 37 itens na tarifa de aplicação, blocos 1 e 2, mais os totais de BE, CVA e FIN do bloco 2 |
| **Familias** | As sete famílias, nas 4 camadas, blocos 1 e 2 |
| **Componentes_BE** | Os mesmos itens na base econômica |
| **Nao_tarifaveis_linha** | Cada linha não tarifável da base, uma a uma |
| **Lotes_sem_calendario** | Mantida pelo layout de SP; vazia enquanto a base não trouxer o lote |
| **APE_Fonte_Incentivada** | Documentos dos livres classificados como APE e/ou fonte incentivada (Passo 4c) |

Nas abas de itens e famílias, as linhas fora da conta entram no fim, com os blocos vazios, marcadas em
`Tarifável?` como **NÃO** (sem tarifa por item) ou **SEM TARIFA** (chave sem tarifa: investigar). A coluna
`Source` identifica a origem da linha: `SAMP` (em SP: `BW` ou `SD`).

In [ ]:
fora["CATEGORIA_TARIFA"] = fora["Ctg.tar."]
fora["OPERACAO"] = fora["Código Arquivo"]
fora["OPERACAO_TXT"] = "SAMP LV - " + fora["TIPO_FATURAMENTO"]
fora["SUBOPERACAO"] = ""
fora["SUBOPERACAO_TXT"] = ""
fora["CLASSE_SUBCLASSE"] = fora["ZnTemp."]

IDENTIFICACAO = ["CHAVE", "CLASSE_PRINCIPAL", "CATEGORIA_TARIFA", "GRP_MONT", "GRP_QUANT",
                 "GRP_QUANT_TXT", "OPERACAO", "OPERACAO_TXT", "SUBOPERACAO", "SUBOPERACAO_TXT",
                 "FLAG_MERCADO", "CLASSE_SUBCLASSE", "TIPO_FATURAMENTO", "LINHAS_ORIGEM",
                 "CONSUMO_TUSD", "CONSUMO_TE", "DEMANDA", "TOTAL_CONSUMO", "TOTAL_DEMANDA",
                 "RECEITA_BRL", "CALCULADO_TA", "DIFERENCA_BRL", "PCT_AJUSTE"]
ROTULO = {"SOURCE": "Source", "CHAVE": "Chave", "CLASSE_PRINCIPAL": "Classe principal",
          "CATEGORIA_TARIFA": "Categoria tarifa", "GRP_MONT": "Grp.estatísts.mont.",
          "GRP_QUANT": "Grp.estatísts.quant.", "GRP_QUANT_TXT": "Grp.estatísts.quant. TEXTO",
          "OPERACAO": "Operação principal", "OPERACAO_TXT": "Operação principal TEXTO",
          "SUBOPERACAO": "Suboperação", "SUBOPERACAO_TXT": "Suboperação TEXTO",
          "FLAG_MERCADO": "Flag Cli.Liv./ Cat.", "CLASSE_SUBCLASSE": "Classe/Sub-classe",
          "TIPO_FATURAMENTO": "Tipo de faturamento",
          "LINHAS_ORIGEM": "Qtd. Cta.contrato", "CONSUMO_TUSD": "Consumo TUSD (kWh)",
          "CONSUMO_TE": "Consumo TE (kWh)", "DEMANDA": "Demanda (kW)",
          "TOTAL_CONSUMO": "TOTAL CONSUMO", "TOTAL_DEMANDA": "TOTAL DEMANDA",
          "RECEITA_BRL": "RECEITA DE FORNECIMENTO",
          "CALCULADO_TA": "VALOR CALCULADO (bloco 1)", "DIFERENCA_BRL": "DIFERENÇA (R$)",
          "PCT_AJUSTE": "DIFERENÇA (%)", "REGIME": "Regime de ajuste",
          "DESC_FIO": "% de desconto no fio",
          "AJUSTE_1A_PASSADA_BRL": "Ajuste 1ª passada (R$)",
          "AJUSTE_2A_PASSADA_BRL": "Ajuste 2ª passada (R$)",
          "FAIXA1_PCT": "% energia na faixa 01 (estimada pela receita)",
          "RECEITA_APE_BRL": "Receita APE incluída (desconto + complemento, R$)",
          "AJUSTE_APE_BRL": "Ajuste regime APE (R$)",
          "AJUSTE_TARIFA_SOCIAL_BRL": "Ajuste Tarifa Social (R$)",
          "TARIFAVEL_TXT": "Tarifável?",
          "ENERGIA_COM_TUSD": "Energia com TUSD (kWh)", "ENERGIA_COM_TE": "Energia com TE (kWh)"}
ENERGIA_DEVIDA = ["ENERGIA_COM_TUSD", "ENERGIA_COM_TE"]
COLS_AJUSTE = ["REGIME", "AJUSTE_TARIFA_SOCIAL_BRL", "RECEITA_APE_BRL", "AJUSTE_APE_BRL", "DESC_FIO",
               "AJUSTE_1A_PASSADA_BRL", "AJUSTE_2A_PASSADA_BRL", "FAIXA1_PCT"]
SOURCE = "SAMP"

def familias_por_camada(bloco):
    """Sum of the items of each family, per layer."""
    return pd.DataFrame({f"{camada} · {fam}": bloco[camada][lista].sum(axis=1)
                         for camada in CAMADAS for fam, lista in FAMILIA.items()})

def monta_aba(colunas_b1, colunas_b2, extra=None, extra_b1=None):
    esq = ident[IDENTIFICACAO].copy()
    esq.insert(0, "SOURCE", SOURCE)
    esq["TARIFAVEL_TXT"] = "SIM"
    corpo = pd.concat([esq.reset_index(drop=True),
                       colunas_b1.add_prefix("[1] ").reset_index(drop=True),
                       ident[COLS_AJUSTE].reset_index(drop=True),
                       colunas_b2.add_prefix("[2] ").reset_index(drop=True)], axis=1)
    if extra is not None:
        corpo = pd.concat([corpo, extra.add_prefix("[2] ").reset_index(drop=True)], axis=1)
    if extra_b1 is not None:
        corpo = pd.concat([corpo, extra_b1.add_prefix("[1] ").reset_index(drop=True)], axis=1)
    corpo = pd.concat([corpo, ident[ENERGIA_DEVIDA].reset_index(drop=True)], axis=1)

    # lines out of the opening, at the end, grouped by kind of entry
    g2 = fora.groupby(["SITUACAO", "CATEGORIA_TARIFA", "FLAG_MERCADO", "GRP_MONT", "TIPO_FATURAMENTO"],
                      dropna=False, sort=False)
    nt = g2.agg(**{c: (c, junta) for c in ["CLASSE_PRINCIPAL", "CLASSE_SUBCLASSE", "GRP_QUANT",
                                           "GRP_QUANT_TXT", "OPERACAO", "OPERACAO_TXT",
                                           "SUBOPERACAO", "SUBOPERACAO_TXT"]})
    nt["LINHAS_ORIGEM"] = g2["Cta.contrato"].count()
    nt["CONSUMO_TUSD"] = g2["QTD_CONSUMO_TUSD"].sum()
    nt["CONSUMO_TE"] = g2["QTD_CONSUMO_TE"].sum()
    nt["DEMANDA"] = g2["DEMANDA_KW"].sum()
    nt["TOTAL_CONSUMO"] = g2["TOTAL_CONSUMO_XLS"].sum()
    nt["TOTAL_DEMANDA"] = g2["TOTAL_DEMANDA_XLS"].sum()
    nt["RECEITA_BRL"] = g2["RECEITA_BRL"].sum()
    nt = nt.reset_index()
    nt["CHAVE"] = np.where(nt["SITUACAO"] == "SEM TARIFA", "(chave sem tarifa na tabela de tarifas)", "")
    nt["TARIFAVEL_TXT"] = nt["SITUACAO"]
    nt["SOURCE"] = SOURCE
    nt = nt.reindex(columns=corpo.columns)
    nt[[c for c in corpo.columns if c.startswith("[")]] = np.nan
    nt[["CALCULADO_TA", "DIFERENCA_BRL", "PCT_AJUSTE"] + COLS_AJUSTE] = np.nan
    return pd.concat([corpo, nt], ignore_index=True).rename(columns=ROTULO)

_totais = pd.DataFrame({f"Total {c}": B2_GRUPO[c][COMPONENTES].sum(axis=1) for c in CAMADAS})
_totais_b1 = pd.DataFrame({f"Total {c}": B1_GRUPO[c][COMPONENTES].sum(axis=1) for c in CAMADAS})
aba_componentes = monta_aba(B1_GRUPO["TA"][COMPONENTES], B2_GRUPO["TA"][COMPONENTES], extra=_totais)
aba_componentes_be = monta_aba(B1_GRUPO["BE"][COMPONENTES], B2_GRUPO["BE"][COMPONENTES], extra=_totais,
                               extra_b1=_totais_b1)
aba_familias = monta_aba(familias_por_camada(B1_GRUPO), familias_por_camada(B2_GRUPO))
print(f"aba Componentes: {aba_componentes.shape[0]} linhas x {aba_componentes.shape[1]} colunas")
print(f"aba Familias   : {aba_familias.shape[0]} linhas x {aba_familias.shape[1]} colunas")

In [ ]:
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

def formata(planilha, n_ident):
    for celula in planilha[1]:
        celula.font = Font(bold=True, size=9, color="FFFFFF")
        celula.fill = PatternFill("solid", fgColor="1F4E79")
        celula.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
    planilha.row_dimensions[1].height = 42
    for col in range(1, planilha.max_column + 1):
        letra = get_column_letter(col)
        planilha.column_dimensions[letra].width = 34 if col == 1 else (22 if col <= n_ident else 16)
        if col > n_ident:
            for linha in range(2, planilha.max_row + 1):
                planilha.cell(row=linha, column=col).number_format = "#,##0.00"
    planilha.auto_filter.ref = f"A1:{get_column_letter(planilha.max_column)}{planilha.max_row}"

LEIA_ME = [
    ("Componentes", "Uma linha por grupo (chave tarifária x classe x operação x tipo de faturamento: FAT ou REFAT). "
     "Colunas \"[1] <item>\": bloco 1 (quantidade x tarifa homologada) na tarifa de aplicação (TA), item a item. "
     "Colunas \"[2] <item>\": bloco 2 (entrega, fecha na receita). \"[2] Total TA/BE/CVA/FIN\": totais do bloco 2 "
     "por camada. Colunas de ajuste: Tarifa Social e regime APE (zeradas nos livres), 1ª passada (fonte incentivada) "
     "e 2ª passada. Últimas colunas, \"Energia com TUSD (kWh)\" e \"Energia com TE (kWh)\": energia sobre a qual "
     "cada parcela é devida (no livre, só TUSD). Source = SAMP (base SAMP LV do sistema comercial).",
     "Conversor do layout Receita Faturada (a adaptar de SP); checks Bl1 x Bl2 e contra o Mercado Receita"),
    ("Componentes_BE", "Mesmas linhas e colunas de Componentes, na base econômica (BE). No fim, os totais por "
     "camada dos dois blocos: \"[2] Total TA/BE/CVA/FIN\" (bloco 2) e \"[1] Total TA/BE/CVA/FIN\" (bloco 1).",
     "Conversor do layout Receita Faturada; check Bl1 x Bl2"),
    ("Familias", "Os itens agrupados nas sete famílias (Fio A, Fio B, encargos, perdas, energia, RI, outros), "
     "por camada (TA, BE, CVA, FIN), nos blocos 1 e 2.", "Checks contra o Mercado Receita"),
    ("Nao_tarifaveis_linha", "As linhas da base sem tarifa homologada (ultrapassagem, bandeira, ICMS, PIS, COFINS "
     "e Outros = TOTAL_FAT menos as demais colunas), uma a uma, com a classe, o tipo de faturamento e a demanda (kW).",
     "Conversor do layout Receita Faturada (colunas Bandeira, UD/ER, Outros, PIS/COFINS, ICMS e DEMANDA "
     "ULTRAPASSAGEM)"),
    ("Lotes_sem_calendario", "Mantida pelo layout de SP. Vazia: a base SAMP ainda não traz o lote de leitura "
     "(todas as linhas usam a tarifa do mês).", "Manutenção do calendário de leitura (quando o lote chegar)"),
    ("APE_Fonte_Incentivada", "Documentos dos clientes livres classificados como APE e/ou fonte incentivada, "
     "inferidos pelo faturamento (Passo 4c): Nº doc., Cta.contrato, classificação e % de desconto de fonte "
     "incentivada (na demanda).",
     "Confirmação com o faturamento; tabela de exceção"),
]
leia_me = pd.DataFrame(LEIA_ME, columns=["Aba", "O que contém", "Onde é usada"])

SAIDA.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(SAIDA, engine="openpyxl") as escritor:
    leia_me.to_excel(escritor, sheet_name="Leia-me", index=False)
    _lm = escritor.book["Leia-me"]
    for _col, _larg in zip("ABC", (34, 110, 80)):
        _lm.column_dimensions[_col].width = _larg
    for _lin in _lm.iter_rows(min_row=1):
        for _cel in _lin:
            _cel.alignment = Alignment(wrap_text=True, vertical="top")
    for _cel in _lm[1]:
        _cel.font = Font(bold=True)
    aba_componentes.to_excel(escritor, sheet_name="Componentes", index=False)
    aba_familias.to_excel(escritor, sheet_name="Familias", index=False)
    aba_componentes_be.to_excel(escritor, sheet_name="Componentes_BE", index=False)
    n_ident = len(IDENTIFICACAO) + 2
    for nome in ("Componentes", "Familias", "Componentes_BE"):
        formata(escritor.book[nome], n_ident)
        planilha = escritor.book[nome]
        for col in range(1, planilha.max_column + 1):
            if planilha.cell(row=1, column=col).value in (ROTULO["PCT_AJUSTE"], ROTULO["DESC_FIO"]):
                for linha in range(2, planilha.max_row + 1):
                    planilha.cell(row=linha, column=col).number_format = "0.00%"

    # non-tariff lines, one row per line of the base
    COLS_NT = {"LINHA_EXCEL": "Linha na base agregada do BW", "FLAG_MERCADO": "Flag Cli.Liv./ Cat.",
               "CLASSE_PRINCIPAL": "Classe principal", "CLASSE_SUBCLASSE": "Classe/Sub-classe",
               "CATEGORIA_TARIFA": "Categoria tarifa", "GRP_MONT": "Grp.estatísts.mont.",
               "GRP_QUANT": "Grp.estatísts.quant.", "OPERACAO": "Operação principal",
               "SUBOPERACAO": "Suboperação", "TIPO_FATURAMENTO": "Tipo de faturamento",
               "DEMANDA_KW": "Demanda (kW)", "RECEITA_BRL": "RECEITA DE FORNECIMENTO",
               "SITUACAO": "Tarifável?"}
    fora[list(COLS_NT)].rename(columns=COLS_NT).to_excel(escritor, sheet_name="Nao_tarifaveis_linha", index=False)
    formata(escritor.book["Nao_tarifaveis_linha"], len(COLS_NT) - 3)

    pd.DataFrame(columns=["Lote no BW (Conjunto de Contrato)", "Linhas na base do BW", "Receita FAT (R$)"]
                 ).to_excel(escritor, sheet_name="Lotes_sem_calendario", index=False)
    formata(escritor.book["Lotes_sem_calendario"], 1)

    # free customers classified as APE / incentivized source (Passo 4c)
    APE_FI.to_excel(escritor, sheet_name="APE_Fonte_Incentivada", index=False)
    formata(escritor.book["APE_Fonte_Incentivada"], 3)
    _ws = escritor.book["APE_Fonte_Incentivada"]
    for linha in range(2, _ws.max_row + 1):
        _ws.cell(row=linha, column=4).number_format = "0%"
print(f"gravado: {SAIDA}")

---
## Passo 16 — Saída para a Regulação

Arquivo `1-Entrega/pxq_regulacao/PxQ_regulacao_<UF>_<ref>.xlsx`, uma aba só, com as mesmas linhas da aba
`Componentes` e as mesmas colunas de SP: identificação, quantidades (energia com TUSD, energia com TE,
demanda), receita e a entrega (bloco 2) item a item nas camadas BE, CVA e FIN. A coluna `Covid Migrantes`
é mantida pelo layout (no RJ e no CE não há esse lançamento na base de livres).

In [ ]:
CAMADAS_REG = ["BE", "CVA", "FIN"]
_reg = monta_aba(pd.DataFrame(index=ident.index),
                 pd.concat([B2_GRUPO[c][COMPONENTES].add_prefix(f"{c} ") for c in CAMADAS_REG], axis=1))
_reg = _reg.rename(columns=lambda n: n[4:] if n.startswith("[2] ") else n)

COLS_ITENS_REG = [f"{c} {i}" for c in CAMADAS_REG for i in COMPONENTES]
COLS_QTD_REG = [ROTULO[c] for c in ["ENERGIA_COM_TUSD", "ENERGIA_COM_TE", "DEMANDA"]]
COLS_ID_REG = ["Source"] + [ROTULO[c] for c in IDENTIFICACAO if c not in
               ("CONSUMO_TUSD", "CONSUMO_TE", "DEMANDA", "TOTAL_CONSUMO", "TOTAL_DEMANDA",
                "RECEITA_BRL", "CALCULADO_TA", "DIFERENCA_BRL", "PCT_AJUSTE")]
aba_regulacao = _reg[COLS_ID_REG + COLS_QTD_REG + [ROTULO["RECEITA_BRL"]] + COLS_ITENS_REG]
COL_COVID_REG = "Covid Migrantes"
aba_regulacao = aba_regulacao.assign(**{COL_COVID_REG: np.nan})

_ta = aba_componentes[[f"[2] {i}" for i in COMPONENTES]].sum(axis=1)
_soma = aba_regulacao[COLS_ITENS_REG].sum(axis=1)
d_rec = abs(aba_regulacao[ROTULO["RECEITA_BRL"]].sum() - aba_componentes[ROTULO["RECEITA_BRL"]].sum())
d_ta = (_soma - _ta).abs().max()
print(f"saida Regulacao: {aba_regulacao.shape[0]} linhas x {aba_regulacao.shape[1]} colunas")
print(f"  receita x aba Componentes ............... diferenca {d_rec:,.6f}")
print(f"  BE + CVA + FIN x TA do bloco 2, por linha  diferenca maxima {d_ta:,.6f}")
if d_rec > 0.01 or d_ta > 0.01:
    raise ValueError("saida Regulacao nao confere com a aba Componentes")

SAIDA_REGULACAO.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(SAIDA_REGULACAO, engine="openpyxl") as escritor:
    aba_regulacao.to_excel(escritor, sheet_name="PxQ_regulacao", index=False)
    _ws = escritor.book["PxQ_regulacao"]
    n_id = len(COLS_ID_REG)
    for celula in _ws[1]:
        celula.font = Font(bold=True, size=9, color="FFFFFF")
        celula.fill = PatternFill("solid", fgColor="1F4E79")
        celula.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
    _ws.row_dimensions[1].height = 42
    for col in range(1, _ws.max_column + 1):
        _ws.column_dimensions[get_column_letter(col)].width = 22 if col <= n_id else 16
        if col > n_id:
            for linha in range(2, _ws.max_row + 1):
                _ws.cell(row=linha, column=col).number_format = "#,##0.00"
    _ws.auto_filter.ref = f"A1:{get_column_letter(_ws.max_column)}{_ws.max_row}"
print(f"gravado: {SAIDA_REGULACAO}")